# Export Classification Reason to Phy

This notebook extracts the main classification reason for each unit from Bombcell results and exports it as a TSV file that can be viewed in Phy's cluster view tab.

The classification reason shows why each unit was classified as GOOD, NOISE, MUA, or NON-SOMA.

## Usage:
1. Set the configuration parameters below (RUN_MODE, TARGET_PROBE, etc.)
2. Run all cells
3. The notebook will create a `cluster_bc_classificationReason.tsv` file in your Kilosort directory
4. Open the data in Phy - you'll see a new "bc_classificationReason" column in the cluster view

### Imports

In [3]:
from pathlib import Path
import re
from typing import Any, Dict
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import sys
import numpy as np
import pandas as pd
import importlib
import subprocess
import os
from dotenv import load_dotenv

plt.rcParams['figure.dpi'] = 120

import bombcell as bc

# ============= Load in configuration and helper functions =============
# Add MOUSE_NAME directory to path
# sys.path.append(str(Path().resolve().parent))  # adds .../mice/Reach15

import sys
from pathlib import Path
# new code
# new code
reach15_dir = Path().resolve().parent  # CWD is .../Reach15/run_bc -> parent is .../Reach15
sys.path.insert(0, str(reach15_dir))   # put it at the FRONT
print("Added:", reach15_dir)
print("sys.path[0:10]:", sys.path[0:10])

print("CWD:", Path().resolve())
print("sys.path[0:5]:", sys.path[0:5])
print("Has Reach15/grant_config.py?:", (Path().resolve().parent / "grant_config.py").exists())
print("Has Reach15/helper_func/grant_config.py?:", (Path().resolve().parent / "helper_func" / "grant_config.py").exists())

import helper_func.prep_data as data_prep
import helper_func.nwb_data_prep as prep
# from helper_func.grant_config import load_grant_config, notebook_runtime_context  # noqa: E402
# from helper_func.post_analysis_setup import load_post_analysis_context  # noqa: E402
from helper_func.grant_config import load_grant_config
from helper_func.post_analysis_setup import load_post_analysis_context
# Always reload local modules so notebook uses latest patched code.
prep = importlib.reload(prep)
# plots = importlib.reload(plots)

print('pca_data_prep path:', Path(prep.__file__).resolve())
if not hasattr(prep, 'extract_probe_letters'):
    raise AttributeError(
        'Loaded pca_data_prep does not expose extract_probe_letters. '
        'Restart kernel and re-run this cell, then confirm it points to master/pca_data_prep.py.'
    )


✅ ipywidgets available - interactive GUI ready
Added: C:\Users\user\Documents\github\bombcell\mice\Reach15
sys.path[0:10]: ['C:\\Users\\user\\Documents\\github\\bombcell\\mice\\Reach15', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\python311.zip', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\DLLs', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\Lib', 'c:\\Users\\user\\anaconda3\\envs\\bombcell', '', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\Lib\\site-packages', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\Lib\\site-packages\\win32', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\Lib\\site-packages\\Pythonwin']
CWD: C:\Users\user\Documents\github\bombcell\mice\Reach15\run_bc
sys.path[0:5]: ['C:\\Users\\user\\Documents\\github\\bombcell\\mice\\Reach15', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\python311.zip', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\DLLs', 'c:\\Users\\user\\anaconda3\\envs\\bombcell\\Lib

### Load data from .env

In [4]:
session_data_dic = prep.load_env()
session_data_dic.keys()

MOUSE loaded: Reach15
-- Behavioral Files --
BEHAVIORAL_FOLDER loaded: grant_reach15_swingDoor-christie
-- Neuropixels Sessions --
Session 1: NP_FILE=Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01 DATE=20260129 SESSION=session003 BOMBCELL=bombcell_batch_20260305_1130
Session 2: NP_FILE=Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32 DATE=20260129 SESSION=session004 BOMBCELL=NA
Session 3: NP_FILE=Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00 DATE=20260201 SESSION=session007 BOMBCELL=bombcell_batch_20260304_1536
-- Config Defaults --
RECORDINGS_ROOT loaded: H:/Grant/Neuropixels/Kilosort_Recordings
OPEN_EPHYS_CONTINUOUS_SUBPATH loaded: Record Node 103/experiment1/recording1/continuous
STRUCTURE_OEBIN_SUBPATH loaded: Record Node 103/experiment1/recording1/structure.oebin
NP20_PROBES loaded: A,C,D


dict_keys(['MOUSE', 'BEHAVIORAL_FOLDER', 'NP_FILE', 'NWB_FILE', 'DATE', 'SESSION', 'BOMBCELL', 'NP_FILE_01', 'NWB_FILE_01', 'DATE_01', 'SESSION_01', 'BOMBCELL_01', 'NP_FILE_02', 'NWB_FILE_02', 'DATE_02', 'SESSION_02', 'BOMBCELL_02', 'RECORDINGS_ROOT', 'OPEN_EPHYS_CONTINUOUS_SUBPATH', 'STRUCTURE_OEBIN_SUBPATH', 'NP20_PROBES', 'RESULTS_PATH', 'RUN_DATES'])

### Configuration

In [ ]:
# Configuration - CHANGE THESE VALUES FOR YOUR DATA
BOMBCELL = session_data_dic['BOMBCELL']
NP_FILE = session_data_dic['NP_FILE']
staging_root = fr'H:\Grant\Neuropixels\Kilosort_Recordings\{NP_FILE}\bombcell\{BOMBCELL}'
TARGET_PROBE = 'A' 

# Optional: Set to True to see detailed information about each unit's classification
VERBOSE = True

In [8]:
# Construct the recording folder path
root_recording_folder = fr"H:\Grant\Neuropixels\Kilosort_Recordings\{session_data_dic['NP_FILE']}"  # Replace with actual root path
recording_folder = os.path.join(root_recording_folder, "Record Node 103", "experiment1", "recording1", "continuous")

#root_behavior_path = fr"G:\Grant\neuropixels\behavioral_recordings\{BEHAVIORAL_FOLDER}"
# Example values (replace as needed)A
root_behavior_path = fr"G:\Grant\behavior_data\DLC_net\{session_data_dic['BEHAVIORAL_FOLDER']}"
behavioral_folder = os.path.join(root_behavior_path, session_data_dic['DATE'] ,session_data_dic['SESSION'])
print(f'behavioral_folder: {behavioral_folder}')
# Check if the recording folder exists
if not os.path.exists(recording_folder):
    raise FileNotFoundError(f"Recording folder not found: {recording_folder}")
else:
    print("Neuropixel Recording folder found:", recording_folder)


behavioral_folder: G:\Grant\behavior_data\DLC_net\grant_reach15_swingDoor-christie\20260129\session003
Neuropixel Recording folder found: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\Record Node 103\experiment1\recording1\continuous


#### =========================================================
## STEP 1: Select Neuropixel session to work with
#### =========================================================

In [9]:
SESSION_TO_ANALYZE = 1

MOUSE, BEHAVIORAL_FOLDER, NP_FILE, NWB_FILE, DATE, SESSION, BOMBCELL = prep.session_to_analyze(
                                                                                        session_data_dic['MOUSE'], session_data_dic['BEHAVIORAL_FOLDER'], 
                                                                                        session_data_dic['NP_FILE'],session_data_dic['NWB_FILE'] ,session_data_dic['DATE'], session_data_dic['SESSION'],session_data_dic['BOMBCELL'],
                                                                                        session_data_dic['NP_FILE_01'], session_data_dic['NWB_FILE_01'], session_data_dic['DATE_01'], session_data_dic['SESSION_01'],session_data_dic['BOMBCELL_01'] ,
                                                                                        session_data_dic['NP_FILE_02'], session_data_dic['NWB_FILE_02'], session_data_dic['DATE_02'], session_data_dic['SESSION_02'],session_data_dic['BOMBCELL_02'],
                                                                                        session_selection=SESSION_TO_ANALYZE
                                                                                    )
                                                               


SESSION SELECTION:

NP_FILE: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
NWB_FILE: NA
DATE: 20260129
SESSION: session003
BEHAVIORAL_FOLDER: grant_reach15_swingDoor-christie
BOMBCELL: bombcell_batch_20260305_1130


### Set and verify bombcell paths using loaded session data from .env

In [10]:
def set_bc_paths(session_data_dic, MOUSE, BEHAVIORAL_FOLDER, DATE, SESSION, SESSION_TO_ANALYZE):
    # SET #1: Path to the NWB file for this session (on the neural data computer)
    NWB_PATH = Path(fr"H:\NWB_OUT\{NWB_FILE}")

    # SET #2: Path to the bombcell root folder for this session (on the neural data computer)
    BOMBCELL_ROOT_FOR_AUTO_BUILD = Path(fr"H:\Grant\Neuropixels\Kilosort_Recordings\{NP_FILE}\bombcell\{BOMBCELL}")

    # SET #3: Session name for labeling plots
    SESSION_NAME = NP_FILE

    # SET #4: Paths to trial index files (behavior acquisition computer)
    baseline_trials_index_path = rf"G:\Grant\behavior_data\DLC_net\{BEHAVIORAL_FOLDER}\videos\{DATE}\christielab\{SESSION}\{DATE}_christielab_{SESSION}_baseline_trial_numbers_tone2_aligned.npy"
    washout_trials_index_path = rf"G:\Grant\behavior_data\DLC_net\{BEHAVIORAL_FOLDER}\videos\{DATE}\christielab\{SESSION}\{DATE}_christielab_{SESSION}_washout_trial_numbers_tone2_aligned.npy"
    optoicalStim_trials_index_path = rf"G:\Grant\behavior_data\DLC_net\{BEHAVIORAL_FOLDER}\videos\{DATE}\christielab\{SESSION}\{DATE}_christielab_{SESSION}_stim_allowed_trial_numbers_tone2_aligned.npy"

    # SET #5: Auto-generate a session-specific config file from .env + selected session
    CONFIG_FILE, _ = prep.build_session_grant_config(
        session_data_dic=session_data_dic,
        session_selection=SESSION_TO_ANALYZE,
        verbose=True,
    )

    for required_path, label in [
        (baseline_trials_index_path, "baseline trials index"),
        (washout_trials_index_path, "washout trials index"),
        (optoicalStim_trials_index_path, "optical stim trials index"),
        # (NWB_PATH, "NWB file"),
        # (BOMBCELL_ROOT_FOR_AUTO_BUILD, "Bombcell root folder"),
        (CONFIG_FILE, "session config file"),
    ]:
        if not Path(required_path).exists():
            raise FileNotFoundError(f"Missing {label}: {required_path}")

    print("All required files/folders found.")
    print(f"NWB file: {NWB_PATH}")
    print(f"Bombcell root folder: {BOMBCELL_ROOT_FOR_AUTO_BUILD}")
    print(f"Session config file: {CONFIG_FILE}")

    return BOMBCELL_ROOT_FOR_AUTO_BUILD, NWB_PATH, CONFIG_FILE, SESSION_NAME


In [11]:
BOMBCELL_ROOT_FOR_AUTO_BUILD, NWB_PATH, CONFIG_FILE, SESSION_NAME = set_bc_paths(
    session_data_dic,
    MOUSE,
    BEHAVIORAL_FOLDER,
    DATE,
    SESSION,
    SESSION_TO_ANALYZE,
)



AUTO-GENERATED SESSION CONFIG

Session selection: 1
Mouse root: C:\Users\user\Documents\github\bombcell\mice\Reach15
Recording name: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
Wrote config: C:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config_20260129_session003.json
All required files/folders found.
NWB file: H:\NWB_OUT\NA
Bombcell root folder: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130
Session config file: C:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config_20260129_session003.json


In [12]:
# Select run mode and target probe (if applicable)
RUN_MODE = 'batch'  # batch | single_probe | np20_rerun
TARGET_PROBE = 'A'  # only used for single_probe
OVERWRITE = True

runner = Path('run_bombcell_unified.py')
cmd = ['python', str(runner), '--config', str(CONFIG_FILE), '--mode', RUN_MODE]
if RUN_MODE == 'single_probe':
    cmd += ['--target-probe', TARGET_PROBE]
if OVERWRITE:
    cmd.append('--overwrite')

print('Running:', ' '.join(cmd))
# subprocess.run(cmd, check=True)
cmd

Running: python run_bombcell_unified.py --config C:\Users\user\Documents\github\bombcell\mice\Reach15\configs\grant_recording_config_20260129_session003.json --mode batch --overwrite


['python',
 'run_bombcell_unified.py',
 '--config',
 'C:\\Users\\user\\Documents\\github\\bombcell\\mice\\Reach15\\configs\\grant_recording_config_20260129_session003.json',
 '--mode',
 'batch',
 '--overwrite']

#### Setup and Load Data

In [13]:
# Load configuration
# cfg = load_grant_config(CONFIG_FILE)


# staging_root, save_subdir = mode_to_roots[RUN_MODE]
ks_dir = Path(root_recording_folder) / f'kilosort4_{TARGET_PROBE}'
save_path = ks_dir / 'bombcell' 

# # Build paths
# ks_dir = Path(staging_root) / f'kilosort4_{TARGET_PROBE}'
# save_path = ks_dir / 'bombcell'

print('Kilosort directory:', ks_dir)
print('Bombcell save path:', save_path)
print()
print('TSV file will be saved to:', ks_dir / 'cluster_bc_classificationReason.tsv')


# Load the config 
ctx = load_post_analysis_context(CONFIG_FILE)

probe_letters = list(ctx['probeLetters'])
config_probe_letters = probe_letters

print('staging_root:', staging_root)
print('probes:', probe_letters)


Kilosort directory: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\kilosort4_A
Bombcell save path: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\kilosort4_A\bombcell

TSV file will be saved to: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\kilosort4_A\cluster_bc_classificationReason.tsv
staging_root: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130
probes: ['A', 'B', 'C', 'D', 'E', 'F']


In [15]:

manually_set_probes = None # -> ["A","B","C"]
manually_set_probes = ['B',"C","D","E",'F']
# --------------------------------------------------------------------------
# This is a workaround if for some reason a probe is giving trouble and you want to skip it , or the probe doesnt exist

if manually_set_probes is not None:
    PROBES = manually_set_probes
    print(f'Using manual probe letters: {PROBES}\n')
    print('To use all probes from config, set manually_set_probes = None')
else:
    PROBES = config_probe_letters
    print(f'Using probe letters from config: {probe_letters}')
# --------------------------------------------------------------------------


Using manual probe letters: ['B', 'C', 'D', 'E', 'F']

To use all probes from config, set manually_set_probes = None


In [16]:


results = []

for probe in PROBES:
    print(f"\n===== Processing Probe {probe} =====")
    ks_dir = Path(staging_root) / f"kilosort4_{probe}"
    save_path = ks_dir / "bombcell"

    print(f"\n===== PROBE {probe} =====")
    print("ks_dir:", ks_dir)
    print("bombcell:", save_path)

    try:
        # 1) Load Bombcell results
        param, quality_metrics, _ = bc.load_bc_results(str(save_path))

        # 2) Unit classifications
        unit_type, unit_type_string = bc.qm.get_quality_unit_type(param, quality_metrics)

        # 3) Build qm_df
        qm_df = pd.DataFrame(quality_metrics).copy()
        qm_df["bombcell_label"] = unit_type_string
        qm_df["unit_index"] = np.arange(len(qm_df))

        # 4) Derive cluster_id (this is the missing piece in your batch cell)
        if "cluster_id" not in qm_df.columns:
            if isinstance(param, dict) and "unique_templates" in param:
                qm_df["cluster_id"] = np.array(param["unique_templates"]).astype(int)
            elif isinstance(quality_metrics, dict) and "phy_clusterID" in quality_metrics:
                qm_df["cluster_id"] = np.array(quality_metrics["phy_clusterID"]).astype(int)
            else:
                qm_df["cluster_id"] = qm_df["unit_index"].astype(int)

        # 5) Minimal export columns for Phy (add more columns here if you want)
        export_df = pd.DataFrame({
            "cluster_id": qm_df["cluster_id"].astype(int),
            "bc_unitType": qm_df["bombcell_label"].astype(str),
        })

        if not export_df["cluster_id"].is_unique:
            raise ValueError("Duplicate cluster_id; Phy merge will be ambiguous")

        out_path = ks_dir / "cluster_bc_unitType.tsv"
        export_df.to_csv(out_path, sep="\t", index=False)
        print("Wrote:", out_path)

        results.append({"probe": probe, "status": "OK", "n_clusters": len(export_df)})

    except Exception as e:
        print("FAILED:", repr(e))
        results.append({"probe": probe, "status": "FAILED", "error": repr(e)})

display(pd.DataFrame(results))



===== Processing Probe B =====

===== PROBE B =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_B
bombcell: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_B\bombcell
Wrote: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_B\cluster_bc_unitType.tsv

===== Processing Probe C =====

===== PROBE C =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_C
bombcell: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_C\bombcell
Wrote: H:\Grant\Neuropixels\Kilosort_Recordings\R

,probe,status,n_clusters
0,B,OK,531
1,C,OK,689
2,D,OK,901
3,E,OK,966
4,F,OK,1486


# Add Brain Region to clusterView

#### Configuration Probes 

In [16]:
# ----------------------------
# ROI / Brain-region configuration
# distance is measured from probe TIP in microns
# ----------------------------
probeA_IP_um = (0, 850)        # Interposed nucleus
probeA_SIM_um = (851, 3250)    # Simplex lobule

ROI_END_UM_BY_PROBE = {
    "A": 3450,
    "B": 950,
    "C": 1800,
    "D": 1400,
    "E": 800,
    "F": 1120,
}

PROBE_TO_REGION = {
    "B": "PG",
    "C": "MoP",
    "D": "VaL",
    "E": "SnR",
    "F": "RN",
}

TIP_POSITION = "min_y"  # min_y is the CORRECT choice for probes. 

#### this cell adds the brain region labels using the above configuration data. It also adds a column with the distance of each unit from the probe tip, which is used for classification.q

In [17]:
# New code
from pathlib import Path
import numpy as np
import pandas as pd



def load_or_init_classification_tsv(ks_dir: Path) -> pd.DataFrame:
    target = ks_dir / "cluster_bc_classificationReason.tsv"

    # If already exists, load it
    if target.exists():
        df = pd.read_csv(target, sep="\t")
        if "cluster_id" not in df.columns:
            raise ValueError(f"{target} exists but missing 'cluster_id'")
        return df

    # Otherwise, build it from the best available source
    cluster_info = ks_dir / "cluster_info.tsv"
    cluster_group = ks_dir / "cluster_group.tsv"

    if cluster_info.exists():
        df = pd.read_csv(cluster_info, sep="\t")
        if "cluster_id" not in df.columns:
            raise ValueError(f"{cluster_info} missing 'cluster_id'")
    elif cluster_group.exists():
        df = pd.read_csv(cluster_group, sep="\t")
        if "cluster_id" not in df.columns:
            raise ValueError(f"{cluster_group} missing 'cluster_id'")
    else:
        # last resort: infer cluster ids from spike_clusters.npy
        sc = np.load(ks_dir / "spike_clusters.npy").astype(np.int64)
        df = pd.DataFrame({"cluster_id": np.unique(sc)})

    # Ensure minimal required column set
    if "cluster_id" not in df.columns:
        raise ValueError("Could not construct a df with 'cluster_id'")

    # Write initial file so downstream steps have a consistent target
    df.to_csv(target, sep="\t", index=False)
    return df


cfg = load_grant_config(CONFIG_FILE)

results = []
errors = []


for probe_letter in PROBES:
    ks_dir = Path(staging_root) / f"kilosort4_{probe_letter}"
    results_dir = Path(staging_root) / "roi_brain_region_results.csv"
    tsv_path = ks_dir / "cluster_bc_classificationReason.tsv"

    print(f"\n===== PROBE {probe_letter} =====")
    print("ks_dir:", ks_dir)

    try:

        # if not tsv_path.exists():
        #     raise FileNotFoundError(f"Missing {tsv_path}. Run the earlier export first.")
        # df = pd.read_csv(tsv_path, sep="\t")

        # New code
        df = load_or_init_classification_tsv(ks_dir)
        cluster_ids = df["cluster_id"].astype(int).to_numpy()
        tsv_path = ks_dir / "cluster_bc_classificationReason.tsv"

        if "cluster_id" not in df.columns:
            raise ValueError(f"{tsv_path} missing 'cluster_id' column")

        cluster_ids = df["cluster_id"].astype(int).to_numpy()

        roi_end_um = ROI_END_UM_BY_PROBE.get(probe_letter, None)
        if roi_end_um is None:
            raise ValueError(f"Probe {probe_letter} missing from ROI_END_UM_BY_PROBE")

        # --- Compute primary channel Y per cluster (same logic as your single-probe cell) ---
        spike_clusters = np.load(ks_dir / "spike_clusters.npy").astype(np.int64)
        spike_templates = np.load(ks_dir / "spike_templates.npy").astype(np.int64)
        templates = np.load(ks_dir / "templates.npy")  # (n_templates, n_time, n_channels)
        channel_map = np.load(ks_dir / "channel_map.npy").astype(np.int64).squeeze()
        channel_positions = np.load(ks_dir / "channel_positions.npy")  # (n_channels_total, 2)

        ptp = templates.max(axis=1) - templates.min(axis=1)  # (n_templates, n_channels)

        primary_y = np.full(cluster_ids.shape, np.nan, dtype=float)

        for i, cid in enumerate(cluster_ids):
            idx = np.where(spike_clusters == cid)[0]
            if idx.size == 0:
                continue
            temps = spike_templates[idx]
            t = np.bincount(temps).argmax()
            c_local = int(np.argmax(ptp[t, :]))
            c_phys = int(channel_map[c_local])
            # New code
            c_phys = int(channel_map[c_local])

            # handle 1-based or off-by-one channel_map
            npos = channel_positions.shape[0]
            if c_phys >= npos:
                if (c_phys - 1) >= 0 and (c_phys - 1) < npos:
                    c_phys = c_phys - 1
                else:
                    raise IndexError(
                        f"channel_map index {c_phys} out of bounds for channel_positions size {npos} "
                        f"(probe {probe_letter}, cluster {cid}, template {t}, c_local {c_local})"
                    )

            primary_y[i] = float(channel_positions[c_phys, 1])
            
        if np.all(np.isnan(primary_y)):
            raise RuntimeError("All primary_y are NaN; check KS files")

        # --- ROI by tip distance ---
        tip_y = np.nanmin(primary_y) if TIP_POSITION == "min_y" else np.nanmax(primary_y)
        dist_um = (primary_y - tip_y) if TIP_POSITION == "min_y" else (tip_y - primary_y)

        roi_label = np.where(dist_um <= float(roi_end_um), "IN_ROI", "OUT_ROI")
        in_roi = roi_label == "IN_ROI"

        # --- Brain region mapping ---
        brain_region = np.full(cluster_ids.shape, np.nan, dtype=object)

        if probe_letter == "A":
            in_ip = (dist_um >= probeA_IP_um[0]) & (dist_um <= probeA_IP_um[1]) & in_roi
            in_sim = (dist_um >= probeA_SIM_um[0]) & (dist_um <= probeA_SIM_um[1]) & in_roi
            brain_region[in_ip] = "IP"
            brain_region[in_sim] = "SIM"
            # 
        else:
            region = PROBE_TO_REGION.get(probe_letter, None)
            if region is None:
                raise ValueError(f"Probe {probe_letter} missing from PROBE_TO_REGION")
            brain_region[in_roi] = region

        # --- Write back (add/overwrite columns) ---
        df["bc_ROI"] = roi_label.astype(str)
        df["Brain_Region"] = brain_region

        df.to_csv(tsv_path, sep="\t", index=False)
        print("Updated:", tsv_path)

        if probe_letter == "A":
            regions_to_summarize = ["SIM", "IP"]
        else:
            regions_to_summarize = [PROBE_TO_REGION[probe_letter]]

        for region in regions_to_summarize:
            in_region = brain_region == region
            out_region = ~in_region

            in_ids = cluster_ids[in_region].astype(int).tolist()
            out_ids = cluster_ids[out_region].astype(int).tolist()

            results.append({
                "probe": probe_letter,
                "brain_region": region,
                "n_clusters": int(len(cluster_ids)),
                "IN_ROI_count": int(np.sum(in_region)),
                "OUT_ROI_count": int(np.sum(out_region)),
                "IN_ROI_unitID": in_ids,
                "OUT_ROI_unitID": out_ids,
            })

    except Exception as e:
        print("FAILED:", repr(e))
        errors.append({"probe": probe_letter, "error": repr(e)})

display(pd.DataFrame(results))
results_df = pd.DataFrame(results)
results_df.to_csv(results_dir, index=False)

if errors:
    display(pd.DataFrame(errors))


===== PROBE B =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_B
Updated: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_B\cluster_bc_classificationReason.tsv

===== PROBE C =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_C
Updated: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_C\cluster_bc_classificationReason.tsv

===== PROBE D =====
ks_dir: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01\bombcell\bombcell_batch_20260305_1130\kilosort4_D
Updated: H:\Grant\Neuropixels\Kilosort_Recordings\Reach15_20260

,probe,brain_region,n_clusters,IN_ROI_count,OUT_ROI_count,IN_ROI_unitID,OUT_ROI_unitID
0,B,PG,531,178,353,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[178, 179, 180, 181, 182, 183, 184, 185, 186, ..."
1,C,MoP,689,689,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
2,D,VaL,901,901,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
3,E,SnR,966,175,791,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[175, 176, 177, 178, 179, 180, 181, 182, 183, ..."
4,F,RN,1486,338,1148,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[335, 336, 339, 340, 341, 342, 344, 345, 346, ..."


### Load in the results df from adding brain regions

In [18]:
results_dir = Path(staging_root) / "roi_brain_region_results.csv"
results_df.to_csv(results_dir, index=False)
results_df

,probe,brain_region,n_clusters,IN_ROI_count,OUT_ROI_count,IN_ROI_unitID,OUT_ROI_unitID
0,B,PG,531,178,353,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[178, 179, 180, 181, 182, 183, 184, 185, 186, ..."
1,C,MoP,689,689,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
2,D,VaL,901,901,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
3,E,SnR,966,175,791,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[175, 176, 177, 178, 179, 180, 181, 182, 183, ..."
4,F,RN,1486,338,1148,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[335, 336, 339, 340, 341, 342, 344, 345, 346, ..."


### Create a dictionary mapping (probe, brain_region) to IN_ROI_unitIDs

In [19]:
results_dir = Path(staging_root) / "roi_brain_region_results.csv"
results_df.to_csv(results_dir, index=False)
results_probe_letters = results_df["probe"].to_list()
results_brain_region = results_df["brain_region"].to_list()
probe_region_pairs = list(zip(results_probe_letters, results_brain_region))
region_inROI_unitIDs = {pair: [] for pair in probe_region_pairs}
for pair in probe_region_pairs:
    probe, brain_region = pair
    region_df = results_df[(results_df["probe"] == probe) & (results_df["brain_region"] == brain_region)]
    if not region_df.empty:
        in_roi_ids = region_df["IN_ROI_unitID"].values[0]
        region_inROI_unitIDs[pair] = in_roi_ids
region_inROI_path =  Path(staging_root)  / "region_inROI_unitIDs.csv"
# save out the region_inROI_unitIDs 

with open(region_inROI_path, "w") as f:
    f.write("probe,brain_region,in_ROI_unitIDs\n")
    for pair, in_roi_ids in region_inROI_unitIDs.items():
        probe, brain_region = pair
        in_roi_ids_str = ";".join(map(str, in_roi_ids))
        f.write(f"{probe},{brain_region},{in_roi_ids_str}\n")

print('probe_letters:', results_probe_letters)
print('brain_regions:', results_brain_region)
print('probe_region_pairs:', probe_region_pairs)
print('region_inROI_unitIDs:', region_inROI_unitIDs)
results_df

probe_letters: ['B', 'C', 'D', 'E', 'F']
brain_regions: ['PG', 'MoP', 'VaL', 'SnR', 'RN']
probe_region_pairs: [('B', 'PG'), ('C', 'MoP'), ('D', 'VaL'), ('E', 'SnR'), ('F', 'RN')]
region_inROI_unitIDs: {('B', 'PG'): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177], ('C

,probe,brain_region,n_clusters,IN_ROI_count,OUT_ROI_count,IN_ROI_unitID,OUT_ROI_unitID
0,B,PG,531,178,353,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[178, 179, 180, 181, 182, 183, 184, 185, 186, ..."
1,C,MoP,689,689,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
2,D,VaL,901,901,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[]
3,E,SnR,966,175,791,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[175, 176, 177, 178, 179, 180, 181, 182, 183, ..."
4,F,RN,1486,338,1148,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[335, 336, 339, 340, 341, 342, 344, 345, 346, ..."


### veiw the unit IDs that are in the ROI for each probe/brain region pair


In [20]:
# veiw the unit IDs that are in the ROI for each probe/brain region pair
for pair, in_roi_ids in region_inROI_unitIDs.items():
    probe, brain_region = pair
    print(f"Probe {probe}, Brain Region {brain_region}, IN_ROI_unitIDs: {in_roi_ids}")

Probe B, Brain Region PG, IN_ROI_unitIDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177]
Probe C, Brain Region MoP, IN_ROI_unitIDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,